# Ibà on N-ATLAS: live demo server

Serves **N-ATLaS** and the **Ibà-ATLAS** adapter with vLLM (OpenAI-compatible API, schema-constrained JSON), runs the Ibà app against it, and opens a temporary public link for recording the demo.

**Kaggle settings:** GPU **T4 x2**, Internet **On**. Secrets (Add-ons → Secrets, ticked for this notebook):
- `HF_TOKEN`: reads N-ATLaS, the adapter and the guideline index
- `NEBIUS_API_KEY`: embeddings for guideline retrieval only (all text generation is N-ATLaS)
- `TAVILY_API_KEY`: live outbreak search (optional)
- `HF_INDEX_REPO`: the private guideline-index dataset, e.g. `jemiridaniel/iba-guideline-index`

Run this in a **separate** notebook session from training. The link dies when the session stops.

In [ ]:
!pip install -q vllm
!pip install -q uv

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(name):
    try:
        return sec.get_secret(name)
    except Exception:
        return None
for k in ("HF_TOKEN", "NEBIUS_API_KEY", "TAVILY_API_KEY", "HF_INDEX_REPO"):
    v = secret(k)
    if v:
        os.environ[k] = v
    print(k, "set" if v else "MISSING")

## 1. Start vLLM (≈5–8 min: download + load across both T4s)

In [ ]:
import subprocess
# Kaggle's kernel refuses "!cmd &", so background servers start via subprocess.Popen.
ADAPTER = "jemiridaniel/Iba-ATLAS-8B-LoRA-Powered-by-Awarri"
vllm = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "NCAIR1/N-ATLaS", "--served-model-name", "NCAIR1/N-ATLaS",
     "--enable-lora", "--max-lora-rank", "16", "--lora-modules", f"iba-atlas={ADAPTER}",
     "--dtype", "half", "--max-model-len", "8192", "--tensor-parallel-size", "2",
     "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
print("vLLM starting, pid", vllm.pid)

In [ ]:
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print("vLLM ready after", i * 10, "s"); break
    except Exception:
        time.sleep(10)
else:
    print("not ready: see the log below")
!tail -n 15 /kaggle/working/vllm.log

## 2. Smoke test: Ibà-ATLAS with schema-constrained JSON

In [ ]:
from openai import OpenAI
llm = OpenAI(base_url="http://localhost:8000/v1", api_key="x")
print([m.id for m in llm.models.list().data])
!git clone -q https://github.com/jemiridaniel/iba-natlas.git /kaggle/working/iba-natlas 2>/dev/null || git -C /kaggle/working/iba-natlas pull -q
import sys; sys.path.insert(0, "/kaggle/working/iba-natlas")
from natlas.prompts import SYSTEM_PROMPT
r = llm.chat.completions.create(model="iba-atlas", temperature=0, max_tokens=200, messages=[
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Pikin 3 years, hot body 4 days, e no fit drink, e dey vomit everything"}])
print(r.choices[0].message.content)

## 3. Start the Ibà app on N-ATLAS

In [ ]:
%cd /kaggle/working/iba-natlas/app
env = """APP_ENV=prod
CHAT_BASE_URL=http://localhost:8000/v1
LLM_STRUCTURED_OUTPUT=json_schema
MODEL_FAST=NCAIR1/N-ATLaS
MODEL_MID=NCAIR1/N-ATLaS
MODEL_REASON=NCAIR1/N-ATLaS
MODEL_ATLAS=iba-atlas
REASONING_REASON=false
MAX_TOKENS_REASONING_OFF=2048
LLM_TIMEOUT_S=120
PATIENT_EXPLANATION=true
MODEL_EMBED=Qwen/Qwen3-Embedding-8B
MAX_SPEND_USD=1.0
MODEL_PRICES={"NCAIR1/N-ATLaS":{"input":0,"output":0},"iba-atlas":{"input":0,"output":0},"Qwen/Qwen3-Embedding-8B":{"input":0.01,"output":0}}
"""
open("/kaggle/working/iba-natlas/app/.env", "w").write(env)
!uv sync -q && uv run -q python -m backend.app.rag.fetch_remote_index
import subprocess
iba = subprocess.Popen(["uv", "run", "-q", "uvicorn", "backend.app.main:app", "--host", "0.0.0.0", "--port", "8080"],
                       cwd="/kaggle/working/iba-natlas/app", stdout=open("/kaggle/working/iba.log", "w"),
                       stderr=subprocess.STDOUT, start_new_session=True)
import time; time.sleep(8)
!curl -s localhost:8080/health; echo; tail -n 5 /kaggle/working/iba.log

## 4. One full case through the pipeline

In [ ]:
import json, requests
body = {"text": "Adult woman 28 years, 62 kg, fever 2 days, RDT positive, eating and drinking well",
        "state": "Lagos", "patient_language": "yo"}
r = requests.post("http://localhost:8080/triage", json=body, timeout=300).json()
print(r["triage_label"], "|", r["triage_rationale"])
print("Ibà-ATLAS:", r.get("atlas"))
print("Yoruba:", (r.get("patient_explanation") or {}).get("text"))
for s in r["decision_trace"]["steps"]:
    print(f"{s['step']:14} {str(s['model']):16} {s['latency_ms']/1000:6.1f}s  {s.get('note') or ''}")

## 5. Public link for the demo video (temporary, dies with the session)

In [ ]:
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared
import subprocess
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8080"],
                          stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
import time, re; time.sleep(10)
print(re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read()))